# Demo Setup

This notebook creates the schema and tables used in **Demo: SQL Editor Guide**.

In [0]:
# Get current user and set up schema name
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"demo_sql_editor_{clean_username}"

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` COMMENT 'Demo schema for SQL Editor exercises'")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

In [0]:
# Create a sales_summary table from TPC-H data
spark.sql(f"""
CREATE OR REPLACE TABLE `{catalog_name}`.`{schema_name}`.sales_summary AS
SELECT 
  n.n_name AS nation,
  r.r_name AS region,
  DATE_TRUNC('month', o.o_orderdate) AS order_month,
  COUNT(*) AS total_orders,
  ROUND(SUM(o.o_totalprice), 2) AS total_revenue,
  ROUND(AVG(o.o_totalprice), 2) AS avg_order_value
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
WHERE o.o_orderdate >= '1996-01-01'
GROUP BY n.n_name, r.r_name, DATE_TRUNC('month', o.o_orderdate)
""")

count = spark.sql("SELECT COUNT(*) FROM sales_summary").first()[0]

In [0]:
# Create a customers table
spark.sql(f"""
CREATE OR REPLACE TABLE `{catalog_name}`.`{schema_name}`.customers AS
SELECT 
  c.c_custkey AS customer_id,
  c.c_name AS customer_name,
  n.n_name AS nation,
  r.r_name AS region,
  c.c_mktsegment AS market_segment,
  c.c_acctbal AS account_balance
FROM samples.tpch.customer c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
""")

count = spark.sql("SELECT COUNT(*) FROM customers").first()[0]

In [0]:
# Add column comments to make schema browser more informative
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN nation COMMENT 'Customer nation name'")
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN region COMMENT 'Geographic region (Africa, America, Asia, Europe, Middle East)'")
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN order_month COMMENT 'First day of the order month'")
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN total_orders COMMENT 'Number of orders placed'")
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN total_revenue COMMENT 'Sum of order totals'")
spark.sql(f"ALTER TABLE sales_summary ALTER COLUMN avg_order_value COMMENT 'Average order value'")

In [0]:
print("SETUP COMPLETE")
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables created:")
print(f"sales_summary  - Monthly sales aggregated by nation/region")
print(f"customers      - Customer details with nation/region/segment")